In [ ]:
import pandas as pd
import io
from google.colab import files

print("Please upload your original Sample - Superstore CSV file:")
uploaded = files.upload()

# Read the uploaded file into a fresh raw DataFrame named 'df'
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"\nRaw dataset successfully loaded! Total rows and columns:{df.shape}")
display(df.head(3))


Please upload your original Sample - Superstore CSV file:


Saving superstore 2.csv to superstore 2 (1).csv

Raw dataset successfully loaded! Total rows and columns:(51290, 27)


,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,00:00.0,CA-2011-130813,...,19,Consumer,00:00.0,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,00:00.0,CA-2011-148614,...,19,Consumer,00:00.0,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,00:00.0,CA-2011-118962,...,21,Consumer,00:00.0,Standard Class,1.81,California,Paper,2011,North America,32


In [ ]:
import sqlite3
# Create a connection to your SQLite database
conn = sqlite3.connect('superstore.db')

# Write the raw dataframe directly into the 'superstore' table (replacing any old tables)
df.to_sql('superstore', conn, if_exists='replace', index=False)

print("Success! Raw dataset loaded into the SQLite database table 'superstore'.")


Success! Raw dataset loaded into the SQLite database table 'superstore'.


In [ ]:
# Query to find sub-categories with negative net profit
loss_query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales),2) AS Total_Sales,
    ROUND(SUM(Profit),2) AS Total_Profit
FROM superstore
GROUP BY "Sub-Category"
HAVING SUM(Profit) < 0
ORDER BY Total_Profit ASC;
"""
loss_results = pd.read_sql(loss_query, conn)
display(loss_results)



,"""Sub-Category""",Total_Sales,Total_Profit


In [ ]:
# 1. Force Sales and Profit columns to be true numbers (Floats)
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce')

# 2. Re-upload the fixed dataframe to SQLite
df.to_sql('superstore', conn, if_exists='replace', index=False)

print("Success! Data types are now forced to numeric and database is refreshed.")

Success! Data types are now forced to numeric and database is refreshed.


In [ ]:
loss_query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM superstore
GROUP BY "Sub-Category"
HAVING SUM(Profit) < 0
ORDER BY Total_Profit ASC;
"""

loss_results = pd.read_sql(loss_query, conn)
display(loss_results)

,"""Sub-Category""",Total_Sales,Total_Profit


In [ ]:
# 1. Group by the exact column name 'Sub.Category' in Pandas
sub_profits = df.groupby('Sub.Category')['Profit'].sum().reset_index()
print("Sub-Category Profits (Pandas Check):")
display(sub_profits.sort_values(by='Profit'))

# 2. Run the SQL query using the correct column name "Sub.Category"
loss_query = """
SELECT
    "Sub.Category",
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM superstore
GROUP BY "Sub.Category"
HAVING SUM(Profit) < 0
ORDER BY Total_Profit ASC;
"""

loss_results = pd.read_sql(loss_query, conn)
print("\nSQL Query Results (Loss-Making Sub-Categories):")
display(loss_results)

Sub-Category Profits (Pandas Check):


,Sub.Category,Profit
16,Tables,-64083.38870
8,Fasteners,11525.42410
10,Labels,15010.51200
15,Supplies,22583.26310
7,Envelopes,29601.11630
9,Furnishings,46967.42550
2,Art,57953.91090
11,Machines,58867.87300
12,Paper,59207.68270
3,Binders,72449.84600



SQL Query Results (Loss-Making Sub-Categories):


,Sub.Category,Total_Sales,Total_Profit
0,Tables,757034.0,-64083.39


In [ ]:
# Save the cleaned dataframe to a new CSV file
cleaned_file_name = "cleaned_superstore_data.csv"
df.to_csv(cleaned_file_name, index=False)

# Automatically download it back to your computer
files.download(cleaned_file_name)
print(f"Success! '{cleaned_file_name}' has been downloaded and is ready for SQL.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Success! 'cleaned_superstore_data.csv' has been downloaded and is ready for SQL.


In [ ]:
from google.colab import files
files.download('superstore.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>